# Module 5 • Neural Networks for Natural Language Processing

# Lesson 30 • Attention Mechanisms for Neural Sequence Models

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 160–200 minutes  
**Execution target:** CPU

---

## Scope

This lesson introduces attention as a dynamic alignment mechanism for neural sequence models. It covers queries, keys, values, score functions, masked softmax, context vectors, additive attention, attentive recurrent decoding, alignment visualization, decoding, evaluation, and failure analysis.

The executable experiment trains a compact English-to-French attentive encoder–decoder locally with PyTorch. It requires no GPU and no external downloads.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain the fixed-vector bottleneck;
- define queries, keys, values, scores, weights, and context vectors;
- implement masked softmax;
- distinguish dot-product, general, and additive attention;
- implement Bahdanau attention;
- integrate attention into an autoregressive decoder;
- train an attentive sequence-to-sequence model;
- visualize source–target alignments;
- evaluate sequence output;
- identify attention failure modes;
- explain why attention is not automatically an explanation;
- discuss Arabic and multilingual alignment issues.

## Table of Contents

1. Why Attention?  
2. Query, Keys, and Values  
3. Attention Scores  
4. Masked Softmax  
5. Context Vectors  
6. Dot-Product, General, and Additive Attention  
7. Attention as Soft Alignment  
8. Manual NumPy Attention  
9. Translation Dataset  
10. Data Splits  
11. Tokenization and Vocabularies  
12. Dynamic Padding  
13. Bidirectional Encoder  
14. Bahdanau Attention  
15. Attentive Decoder  
16. Complete Model  
17. Shape and Mask Inspection  
18. Training  
19. Learning Curves  
20. Greedy Decoding  
21. Alignment Visualization  
22. Attention Entropy  
23. Evaluation  
24. Error Analysis  
25. Failure Modes  
26. Computational Cost  
27. Attention and Interpretability  
28. Arabic and Multilingual Considerations  
29. Reproducibility  
30. Knowledge Check  
31. Exercises  
32. Summary and Next Lesson

# 1. Why Attention?

A basic recurrent encoder–decoder compresses the complete source sequence into one final vector. This creates a **fixed-vector bottleneck**.

Attention lets the decoder consult every encoder state at every target step:

```text
encoder states: h1, h2, h3, ..., hT
                 ↘  ↓  ↙
            attention weights
                   ↓
             context vector
                   ↓
                decoder
```

In [ ]:
import copy
import math
import random
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset

architecture_comparison = pd.DataFrame(
    [
        ("Fixed-vector encoder–decoder", "one final source vector", "static"),
        ("Attentive encoder–decoder", "all encoder states", "target-specific"),
    ],
    columns=["Architecture", "Source access", "Context"],
)
architecture_comparison

# 2. Query, Keys, and Values

Attention uses three functional roles:

- **Query:** what the decoder currently needs.
- **Keys:** source representations used for scoring.
- **Values:** source representations combined into the context vector.

In recurrent translation, the decoder hidden state is the query, while encoder states are usually both keys and values.

In [ ]:
qkv_shapes = pd.DataFrame(
    [
        ("Query", "decoder state", "(B, D)"),
        ("Keys", "encoder states", "(B, T, E)"),
        ("Values", "encoder states", "(B, T, E)"),
        ("Weights", "normalized scores", "(B, T)"),
        ("Context", "weighted value sum", "(B, E)"),
    ],
    columns=["Object", "Role", "Shape"],
)
qkv_shapes

# 3. Attention Scores

A score function measures compatibility between the query and each key. Scores become probabilities only after normalization.

Common score functions include:

- dot product;
- general or bilinear attention;
- additive or Bahdanau attention.

# 4. Masked Softmax

Padding positions must receive zero attention probability. A mask replaces padded scores with a very negative value before softmax.

In [ ]:
def masked_softmax_numpy(scores: np.ndarray, mask: np.ndarray) -> np.ndarray:
    masked_scores = np.where(mask, scores, -1e9)
    shifted = masked_scores - masked_scores.max(axis=-1, keepdims=True)
    exponentials = np.exp(shifted) * mask.astype(float)
    return exponentials / np.maximum(exponentials.sum(axis=-1, keepdims=True), 1e-12)

scores = np.array([[1.2, 0.4, 2.1, -0.5], [0.2, 1.4, 0.3, 1.0]])
mask = np.array([[True, True, True, False], [True, True, False, False]])
weights = masked_softmax_numpy(scores, mask)

print(weights)
print("Row sums:", weights.sum(axis=1))

# 5. Context Vectors

The context vector is a weighted sum of values:

$$
c = \sum_{i=1}^{T} lpha_i v_i
$$

The decoder receives a different context at each target step.

In [ ]:
values = np.array(
    [
        [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [5.0, 5.0]],
        [[2.0, 0.0], [0.0, 2.0], [9.0, 9.0], [9.0, 9.0]],
    ]
)
contexts = np.einsum("bt,btd->bd", weights, values)
contexts

# 6. Dot-Product, General, and Additive Attention

**Dot product**

$$score(q,k_i)=q^Tk_i$$

**General attention**

$$score(q,k_i)=q^TWk_i$$

**Additive attention**

$$e_{t,i}=v^T	anh(W_hh_i+W_ss_{t-1})$$

Additive attention supports different encoder and decoder dimensions and is used in the executable model.

In [ ]:
rng = np.random.default_rng(42)
query = np.array([[0.7, -0.2, 0.5]])
keys = np.array([[[0.8, -0.1, 0.4], [0.1, 0.9, -0.3], [0.6, -0.2, 0.7]]])

dot_scores = np.einsum("bd,btd->bt", query, keys)
projection = rng.normal(0.0, 0.2, size=(3, 3))
general_scores = np.einsum("bd,btd->bt", query, keys @ projection)

pd.DataFrame({"dot": dot_scores[0], "general": general_scores[0]})

# 7. Attention as Soft Alignment

Each decoder step produces a probability distribution over source positions. The distribution can be viewed operationally as a **soft alignment** between a target step and source tokens.

Soft alignment may distribute probability across several positions rather than selecting one hard link.

# 8. Manual NumPy Additive Attention

In [ ]:
batch_size = 2
source_length = 5
encoder_dim = 6
decoder_dim = 4
attention_dim = 7

encoder_states = rng.normal(size=(batch_size, source_length, encoder_dim))
decoder_state = rng.normal(size=(batch_size, decoder_dim))
W_encoder = rng.normal(0.0, 0.2, size=(encoder_dim, attention_dim))
W_decoder = rng.normal(0.0, 0.2, size=(decoder_dim, attention_dim))
v = rng.normal(0.0, 0.2, size=(attention_dim,))

energy = np.tanh(
    encoder_states @ W_encoder
    + (decoder_state @ W_decoder)[:, None, :]
)
manual_scores = energy @ v
manual_mask = np.array(
    [[True, True, True, True, False], [True, True, True, False, False]]
)
manual_weights = masked_softmax_numpy(manual_scores, manual_mask)
manual_context = np.einsum("bt,btd->bd", manual_weights, encoder_states)

print("Scores:", manual_scores.shape)
print("Weights:", manual_weights.shape)
print("Context:", manual_context.shape)

# 9. Translation Dataset

The controlled dataset maps English commands to deterministic French commands. It requires lexical translation, article selection, adjective agreement, and adjective reordering.

In [ ]:
verbs = {
    "open": "ouvre",
    "close": "ferme",
    "find": "trouve",
    "take": "prends",
    "move": "deplace",
}

nouns = {
    "door": {"article": "la", "translation": "porte", "gender": "f"},
    "window": {"article": "la", "translation": "fenetre", "gender": "f"},
    "box": {"article": "la", "translation": "boite", "gender": "f"},
    "book": {"article": "le", "translation": "livre", "gender": "m"},
    "key": {"article": "la", "translation": "cle", "gender": "f"},
}

modifiers = {
    "red": {"position": "after", "m": "rouge", "f": "rouge"},
    "blue": {"position": "after", "m": "bleu", "f": "bleue"},
    "green": {"position": "after", "m": "vert", "f": "verte"},
    "yellow": {"position": "after", "m": "jaune", "f": "jaune"},
    "small": {"position": "before", "m": "petit", "f": "petite"},
    "big": {"position": "before", "m": "grand", "f": "grande"},
}


def build_translation(verb: str, noun: str, modifier: str) -> tuple[str, str]:
    noun_info = nouns[noun]
    modifier_info = modifiers[modifier]
    translated_modifier = modifier_info[noun_info["gender"]]
    source = f"{verb} the {modifier} {noun}"

    if modifier_info["position"] == "before":
        target = (
            f"{verbs[verb]} {noun_info['article']} "
            f"{translated_modifier} {noun_info['translation']}"
        )
    else:
        target = (
            f"{verbs[verb]} {noun_info['article']} "
            f"{noun_info['translation']} {translated_modifier}"
        )

    return source, target

records = []
for verb in verbs:
    for noun in nouns:
        for modifier in modifiers:
            source, target = build_translation(verb, noun, modifier)
            records.append(
                {
                    "source": source,
                    "target": target,
                    "verb": verb,
                    "noun": noun,
                    "modifier": modifier,
                }
            )

dataset = pd.DataFrame(records)
print("Sentence pairs:", len(dataset))
dataset.sample(8, random_state=42)[["source", "target"]].reset_index(drop=True)

# 10. Data Splits

In [ ]:
train_frame, test_frame = train_test_split(
    dataset, test_size=0.15, random_state=42
)
train_frame, validation_frame = train_test_split(
    train_frame, test_size=0.1765, random_state=42
)

train_frame = train_frame.reset_index(drop=True)
validation_frame = validation_frame.reset_index(drop=True)
test_frame = test_frame.reset_index(drop=True)

pd.Series(
    {
        "training": len(train_frame),
        "validation": len(validation_frame),
        "test": len(test_frame),
    }
)

In [ ]:
print("Verb coverage:", set(train_frame["verb"]) == set(verbs))
print("Noun coverage:", set(train_frame["noun"]) == set(nouns))
print("Modifier coverage:", set(train_frame["modifier"]) == set(modifiers))

# 11. Tokenization and Vocabularies

Source and target languages use separate vocabularies. PAD, UNK, BOS, and EOS control batching and generation.

In [ ]:
TOKEN_PATTERN = re.compile(r"\w+(?:[-']\w+)*", flags=re.UNICODE)


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text.lower())


PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
BOS_TOKEN = "<BOS>"
EOS_TOKEN = "<EOS>"


class Vocabulary:
    def __init__(self, texts, include_bos: bool):
        counts = Counter(token for text in texts for token in tokenize(text))
        special = [PAD_TOKEN, UNK_TOKEN]
        if include_bos:
            special.append(BOS_TOKEN)
        special.append(EOS_TOKEN)
        self.index_to_token = special + sorted(counts)
        self.token_to_index = {
            token: index for index, token in enumerate(self.index_to_token)
        }
        self.pad_id = self.token_to_index[PAD_TOKEN]
        self.unk_id = self.token_to_index[UNK_TOKEN]
        self.eos_id = self.token_to_index[EOS_TOKEN]
        self.bos_id = self.token_to_index.get(BOS_TOKEN)

    def __len__(self):
        return len(self.index_to_token)

    def encode(self, text: str, add_bos: bool = False) -> list[int]:
        ids = []
        if add_bos:
            if self.bos_id is None:
                raise ValueError("BOS is unavailable in this vocabulary")
            ids.append(self.bos_id)
        ids.extend(
            self.token_to_index.get(token, self.unk_id)
            for token in tokenize(text)
        )
        ids.append(self.eos_id)
        return ids

    def decode(self, ids, stop_at_eos: bool = True) -> list[str]:
        tokens = []
        for token_id in ids:
            token = self.index_to_token[int(token_id)]
            if stop_at_eos and token == EOS_TOKEN:
                break
            if token not in {PAD_TOKEN, BOS_TOKEN}:
                tokens.append(token)
        return tokens


source_vocabulary = Vocabulary(train_frame["source"], include_bos=False)
target_vocabulary = Vocabulary(train_frame["target"], include_bos=True)

print("Source vocabulary:", len(source_vocabulary))
print("Target vocabulary:", len(target_vocabulary))

# 12. Dataset and Dynamic Padding

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return {
            "source_ids": torch.tensor(
                source_vocabulary.encode(row["source"]), dtype=torch.long
            ),
            "target_ids": torch.tensor(
                target_vocabulary.encode(row["target"], add_bos=True),
                dtype=torch.long,
            ),
            "source_text": row["source"],
            "target_text": row["target"],
        }


def collate_batch(batch):
    source_lengths = torch.tensor(
        [len(item["source_ids"]) for item in batch], dtype=torch.long
    )
    target_lengths = torch.tensor(
        [len(item["target_ids"]) for item in batch], dtype=torch.long
    )

    source_ids = torch.full(
        (len(batch), int(source_lengths.max())),
        source_vocabulary.pad_id,
        dtype=torch.long,
    )
    target_ids = torch.full(
        (len(batch), int(target_lengths.max())),
        target_vocabulary.pad_id,
        dtype=torch.long,
    )

    for row, item in enumerate(batch):
        source_ids[row, : len(item["source_ids"])] = item["source_ids"]
        target_ids[row, : len(item["target_ids"])] = item["target_ids"]

    return {
        "source_ids": source_ids,
        "source_lengths": source_lengths,
        "target_ids": target_ids,
        "target_lengths": target_lengths,
        "source_texts": [item["source_text"] for item in batch],
        "target_texts": [item["target_text"] for item in batch],
    }


train_dataset = TranslationDataset(train_frame)
validation_dataset = TranslationDataset(validation_frame)
test_dataset = TranslationDataset(test_frame)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_batch,
    generator=torch.Generator().manual_seed(42),
)
validation_loader = DataLoader(
    validation_dataset, batch_size=16, shuffle=False, collate_fn=collate_batch
)
test_loader = DataLoader(
    test_dataset, batch_size=16, shuffle=False, collate_fn=collate_batch
)

sample_batch = next(iter(train_loader))
print("Source batch:", sample_batch["source_ids"].shape)
print("Target batch:", sample_batch["target_ids"].shape)

# 13. Bidirectional Encoder

The encoder returns one contextual vector per source position and a bridged initial decoder state.

In [ ]:
class BidirectionalEncoder(nn.Module):
    def __init__(
        self,
        vocabulary_size: int,
        embedding_dim: int,
        encoder_hidden_dim: int,
        decoder_hidden_dim: int,
        padding_id: int,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.embedding = nn.Embedding(
            vocabulary_size, embedding_dim, padding_idx=padding_id
        )
        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(
            embedding_dim,
            encoder_hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.bridge = nn.Linear(2 * encoder_hidden_dim, decoder_hidden_dim)

    def forward(self, source_ids, source_lengths):
        embedded = self.dropout(self.embedding(source_ids))
        packed = pack_padded_sequence(
            embedded,
            source_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_outputs, hidden = self.gru(packed)
        outputs, _ = pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=source_ids.size(1),
        )
        combined = torch.cat([hidden[-2], hidden[-1]], dim=1)
        decoder_initial = torch.tanh(self.bridge(combined))
        return outputs, decoder_initial

# 14. Bahdanau Additive Attention

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, encoder_dim: int, decoder_dim: int, attention_dim: int):
        super().__init__()
        self.encoder_projection = nn.Linear(encoder_dim, attention_dim, bias=False)
        self.decoder_projection = nn.Linear(decoder_dim, attention_dim, bias=False)
        self.energy_projection = nn.Linear(attention_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, source_mask):
        encoder_energy = self.encoder_projection(encoder_outputs)
        decoder_energy = self.decoder_projection(decoder_hidden).unsqueeze(1)
        scores = self.energy_projection(
            torch.tanh(encoder_energy + decoder_energy)
        ).squeeze(-1)
        scores = scores.masked_fill(~source_mask, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, weights

# 15. Attentive Decoder

At every step, the decoder:

1. embeds the previous target token;
2. calculates source attention;
3. creates a context vector;
4. updates its hidden state;
5. predicts the next target token.

In [ ]:
class AttentiveDecoder(nn.Module):
    def __init__(
        self,
        vocabulary_size: int,
        embedding_dim: int,
        encoder_dim: int,
        decoder_hidden_dim: int,
        attention_dim: int,
        padding_id: int,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.embedding = nn.Embedding(
            vocabulary_size, embedding_dim, padding_idx=padding_id
        )
        self.dropout = nn.Dropout(dropout)
        self.attention = BahdanauAttention(
            encoder_dim, decoder_hidden_dim, attention_dim
        )
        self.gru_cell = nn.GRUCell(
            embedding_dim + encoder_dim, decoder_hidden_dim
        )
        self.output_layer = nn.Linear(
            decoder_hidden_dim + encoder_dim + embedding_dim,
            vocabulary_size,
        )

    def forward_step(
        self, token_ids, decoder_hidden, encoder_outputs, source_mask
    ):
        embedded = self.dropout(self.embedding(token_ids))
        context, weights = self.attention(
            decoder_hidden, encoder_outputs, source_mask
        )
        decoder_hidden = self.gru_cell(
            torch.cat([embedded, context], dim=1), decoder_hidden
        )
        logits = self.output_layer(
            torch.cat([decoder_hidden, context, embedded], dim=1)
        )
        return logits, decoder_hidden, weights

# 16. Complete Attentive Sequence-to-Sequence Model

In [ ]:
class AttentiveSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder, source_pad_id: int):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.source_pad_id = source_pad_id

    def encode(self, source_ids, source_lengths):
        encoder_outputs, decoder_hidden = self.encoder(
            source_ids, source_lengths
        )
        source_mask = source_ids != self.source_pad_id
        return encoder_outputs, decoder_hidden, source_mask

    def forward(
        self,
        source_ids,
        source_lengths,
        target_ids,
        teacher_forcing_ratio: float,
    ):
        encoder_outputs, decoder_hidden, source_mask = self.encode(
            source_ids, source_lengths
        )
        batch_size = source_ids.size(0)
        target_length = target_ids.size(1)
        vocabulary_size = self.decoder.output_layer.out_features

        all_logits = torch.zeros(
            batch_size,
            target_length - 1,
            vocabulary_size,
            device=source_ids.device,
        )
        all_attention = torch.zeros(
            batch_size,
            target_length - 1,
            source_ids.size(1),
            device=source_ids.device,
        )

        decoder_input = target_ids[:, 0]
        for time_step in range(1, target_length):
            logits, decoder_hidden, weights = self.decoder.forward_step(
                decoder_input,
                decoder_hidden,
                encoder_outputs,
                source_mask,
            )
            all_logits[:, time_step - 1, :] = logits
            all_attention[:, time_step - 1, :] = weights
            predicted = logits.argmax(dim=1)
            decoder_input = (
                target_ids[:, time_step]
                if random.random() < teacher_forcing_ratio
                else predicted
            )

        return {"logits": all_logits, "attention": all_attention}

In [ ]:
DEVICE = torch.device("cpu")
ENCODER_HIDDEN_DIM = 24
DECODER_HIDDEN_DIM = 48

torch.manual_seed(42)
encoder = BidirectionalEncoder(
    vocabulary_size=len(source_vocabulary),
    embedding_dim=32,
    encoder_hidden_dim=ENCODER_HIDDEN_DIM,
    decoder_hidden_dim=DECODER_HIDDEN_DIM,
    padding_id=source_vocabulary.pad_id,
)
decoder = AttentiveDecoder(
    vocabulary_size=len(target_vocabulary),
    embedding_dim=32,
    encoder_dim=2 * ENCODER_HIDDEN_DIM,
    decoder_hidden_dim=DECODER_HIDDEN_DIM,
    attention_dim=40,
    padding_id=target_vocabulary.pad_id,
)
model = AttentiveSeq2Seq(
    encoder, decoder, source_vocabulary.pad_id
).to(DEVICE)

print(
    "Trainable parameters:",
    sum(parameter.numel() for parameter in model.parameters()),
)

# 17. Shape and Mask Inspection

In [ ]:
with torch.no_grad():
    inspection_output = model(
        sample_batch["source_ids"].to(DEVICE),
        sample_batch["source_lengths"].to(DEVICE),
        sample_batch["target_ids"].to(DEVICE),
        teacher_forcing_ratio=1.0,
    )

print("Logits:", inspection_output["logits"].shape)
print("Attention:", inspection_output["attention"].shape)

In [ ]:
source_ids = sample_batch["source_ids"].to(DEVICE)
valid_mask = source_ids != source_vocabulary.pad_id
attention_tensor = inspection_output["attention"]

padding_mass = attention_tensor.masked_select(~valid_mask[:, None, :]).sum()
row_sums = attention_tensor.sum(dim=2)

print("Attention mass on padding:", float(padding_mass))
print("Mean attention row sum:", float(row_sums.mean()))

# 18. Training

Sequence cross-entropy ignores target padding. Teacher forcing decreases gradually. Gradient clipping and early stopping stabilize training.

In [ ]:
loss_function = nn.CrossEntropyLoss(ignore_index=target_vocabulary.pad_id)


def sequence_loss(logits, target_ids):
    expected = target_ids[:, 1:]
    return loss_function(
        logits.reshape(-1, logits.size(-1)),
        expected.reshape(-1),
    )


def teacher_forcing_schedule(epoch: int, total_epochs: int) -> float:
    progress = epoch / max(total_epochs - 1, 1)
    return 1.0 + progress * (0.45 - 1.0)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def evaluate_validation_loss(model, loader):
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in loader:
            output = model(
                batch["source_ids"].to(DEVICE),
                batch["source_lengths"].to(DEVICE),
                batch["target_ids"].to(DEVICE),
                teacher_forcing_ratio=0.0,
            )
            losses.append(
                float(
                    sequence_loss(
                        output["logits"], batch["target_ids"].to(DEVICE)
                    ).item()
                )
            )
    return float(np.mean(losses))

In [ ]:
def train_attention_model(
    model,
    epochs: int = 55,
    learning_rate: float = 0.006,
    clip_norm: float = 5.0,
    patience: int = 10,
):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    best_state = copy.deepcopy(model.state_dict())
    best_validation_loss = float("inf")
    epochs_without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()
        ratio = teacher_forcing_schedule(epoch, epochs)
        training_losses = []
        gradient_norms = []

        for batch in train_loader:
            optimizer.zero_grad()
            source_ids = batch["source_ids"].to(DEVICE)
            source_lengths = batch["source_lengths"].to(DEVICE)
            target_ids = batch["target_ids"].to(DEVICE)

            output = model(
                source_ids,
                source_lengths,
                target_ids,
                teacher_forcing_ratio=ratio,
            )
            loss = sequence_loss(output["logits"], target_ids)
            loss.backward()
            gradient_norm = clip_grad_norm_(model.parameters(), max_norm=clip_norm)
            optimizer.step()

            training_losses.append(float(loss.item()))
            gradient_norms.append(float(gradient_norm))

        validation_loss = evaluate_validation_loss(model, validation_loader)
        history.append(
            {
                "epoch": epoch,
                "training_loss": float(np.mean(training_losses)),
                "validation_loss": validation_loss,
                "teacher_forcing_ratio": ratio,
                "mean_gradient_norm": float(np.mean(gradient_norms)),
            }
        )

        if validation_loss < best_validation_loss - 1e-5:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


set_seed(42)
trained_model, training_history = train_attention_model(model)
print("Epochs completed:", len(training_history))
print("Best validation loss:", round(training_history["validation_loss"].min(), 4))

# 19. Learning Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(training_history["epoch"], training_history["training_loss"], label="Training loss")
plt.plot(training_history["epoch"], training_history["validation_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Attentive Seq2Seq Learning Curves")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(training_history["epoch"], training_history["mean_gradient_norm"])
plt.xlabel("Epoch")
plt.ylabel("Mean pre-clipping gradient norm")
plt.title("Attention Model Gradient Norms")
plt.tight_layout()
plt.show()

# 20. Greedy Attention Decoding

In [ ]:
def greedy_decode(model, source_text: str, maximum_length: int = 10):
    model.eval()
    source_token_ids = source_vocabulary.encode(source_text)
    source_ids = torch.tensor([source_token_ids], dtype=torch.long, device=DEVICE)
    source_lengths = torch.tensor([len(source_token_ids)], dtype=torch.long, device=DEVICE)

    with torch.no_grad():
        encoder_outputs, decoder_hidden, source_mask = model.encode(
            source_ids, source_lengths
        )
        decoder_input = torch.tensor(
            [target_vocabulary.bos_id], dtype=torch.long, device=DEVICE
        )

        generated_ids = []
        token_probabilities = []
        attention_rows = []

        for _ in range(maximum_length):
            logits, decoder_hidden, weights = model.decoder.forward_step(
                decoder_input,
                decoder_hidden,
                encoder_outputs,
                source_mask,
            )
            probabilities = torch.softmax(logits, dim=1)
            predicted_id = int(probabilities.argmax(dim=1).item())
            generated_ids.append(predicted_id)
            token_probabilities.append(float(probabilities[0, predicted_id].item()))
            attention_rows.append(weights[0].cpu().numpy())

            if predicted_id == target_vocabulary.eos_id:
                break

            decoder_input = torch.tensor(
                [predicted_id], dtype=torch.long, device=DEVICE
            )

    return {
        "translation": " ".join(target_vocabulary.decode(generated_ids)),
        "generated_ids": generated_ids,
        "probabilities": token_probabilities,
        "attention": np.asarray(attention_rows),
        "source_ids": source_token_ids,
    }


greedy_decode(trained_model, "open the red door")["translation"]

# 21. Alignment Visualization

In [ ]:
visualization_source = "move the small window"
visualization_result = greedy_decode(trained_model, visualization_source)

source_labels = [
    source_vocabulary.index_to_token[token_id]
    for token_id in visualization_result["source_ids"]
]
target_labels = [
    target_vocabulary.index_to_token[token_id]
    for token_id in visualization_result["generated_ids"]
]
attention_matrix = visualization_result["attention"]

plt.figure(figsize=(9, 6))
plt.imshow(attention_matrix, aspect="auto")
plt.xticks(range(len(source_labels)), source_labels, rotation=45, ha="right")
plt.yticks(range(len(target_labels)), target_labels)
plt.xlabel("Source tokens")
plt.ylabel("Generated target tokens")
plt.title("Learned Source–Target Attention")
plt.colorbar()
plt.tight_layout()
plt.show()

The heatmap shows relative model focus. It should be interpreted comparatively rather than as proof of human-like reasoning.

# 22. Attention Entropy

Entropy measures concentration. Low entropy is concentrated; high entropy is diffuse.

In [ ]:
def attention_entropy(attention_weights: np.ndarray) -> np.ndarray:
    clipped = np.clip(attention_weights, 1e-12, 1.0)
    return -np.sum(clipped * np.log(clipped), axis=1)

pd.DataFrame(
    {
        "target_token": target_labels,
        "attention_entropy": attention_entropy(attention_matrix),
    }
)

Entropy does not determine whether an alignment is linguistically correct.

# 23. Evaluation

The notebook reports exact match, positional token accuracy, and an educational BLEU-like score.

In [ ]:
def positional_token_accuracy(reference: str, prediction: str) -> float:
    reference_tokens = tokenize(reference)
    prediction_tokens = tokenize(prediction)
    denominator = max(len(reference_tokens), len(prediction_tokens), 1)
    matches = sum(
        left == right
        for left, right in zip(reference_tokens, prediction_tokens)
    )
    return matches / denominator


def extract_ngrams(tokens: list[str], order: int) -> Counter:
    return Counter(
        tuple(tokens[index : index + order])
        for index in range(len(tokens) - order + 1)
    )


def sentence_bleu_like(reference: str, prediction: str, maximum_order: int = 4) -> float:
    reference_tokens = tokenize(reference)
    prediction_tokens = tokenize(prediction)
    if not prediction_tokens:
        return 0.0

    precisions = []
    for order in range(1, maximum_order + 1):
        predicted_ngrams = extract_ngrams(prediction_tokens, order)
        reference_ngrams = extract_ngrams(reference_tokens, order)
        clipped = sum(
            min(count, reference_ngrams[ngram])
            for ngram, count in predicted_ngrams.items()
        )
        total = sum(predicted_ngrams.values())
        precisions.append((clipped + 1.0) / (total + 1.0))

    reference_length = len(reference_tokens)
    prediction_length = len(prediction_tokens)
    brevity_penalty = (
        1.0
        if prediction_length > reference_length
        else math.exp(1.0 - reference_length / max(prediction_length, 1))
    )
    geometric_mean = math.exp(
        sum(math.log(max(value, 1e-12)) for value in precisions)
        / maximum_order
    )
    return brevity_penalty * geometric_mean

In [ ]:
def evaluate_translations(model, frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for row in frame.itertuples(index=False):
        result = greedy_decode(model, row.source)
        prediction = result["translation"]
        rows.append(
            {
                "source": row.source,
                "reference": row.target,
                "prediction": prediction,
                "exact_match": prediction.strip() == row.target.strip(),
                "token_accuracy": positional_token_accuracy(row.target, prediction),
                "bleu_like": sentence_bleu_like(row.target, prediction),
                "mean_confidence": float(np.mean(result["probabilities"])),
                "mean_attention_entropy": float(
                    np.mean(attention_entropy(result["attention"]))
                ),
            }
        )
    return pd.DataFrame(rows)


test_results = evaluate_translations(trained_model, test_frame)
print("Exact match:", round(test_results["exact_match"].mean(), 3))
print("Token accuracy:", round(test_results["token_accuracy"].mean(), 3))
print("BLEU-like:", round(test_results["bleu_like"].mean(), 3))

Formal translation studies should use established implementations of BLEU, chrF, COMET, and other metrics.

# 24. Error Analysis

In [ ]:
test_results.sort_values(
    ["exact_match", "bleu_like", "mean_confidence"],
    ascending=[True, True, False],
).reset_index(drop=True)

In [ ]:
def categorize_error(reference: str, prediction: str, exact: bool) -> str:
    if exact:
        return "correct"
    reference_tokens = tokenize(reference)
    prediction_tokens = tokenize(prediction)
    if len(prediction_tokens) < len(reference_tokens):
        return "too short"
    if len(prediction_tokens) > len(reference_tokens):
        return "too long"
    if set(prediction_tokens) == set(reference_tokens):
        return "word order"
    return "lexical or agreement error"


test_results["error_type"] = [
    categorize_error(reference, prediction, exact)
    for reference, prediction, exact in zip(
        test_results["reference"],
        test_results["prediction"],
        test_results["exact_match"],
    )
]

test_results["error_type"].value_counts()

# 25. Alignment Failure Modes

Common alignment problems include:

- diffuse attention;
- repeated focus on one source token;
- skipped source information;
- incorrect adjective–noun alignment;
- excessive attention to EOS;
- stable but linguistically wrong patterns.

In [ ]:
alignment_failures = pd.DataFrame(
    [
        ("Diffuse", "high entropy across many positions"),
        ("Collapsed", "one position dominates many target steps"),
        ("Skipped", "important source token receives little mass"),
        ("Misaligned", "focuses on the wrong lexical item"),
        ("EOS-biased", "source EOS attracts excessive attention"),
    ],
    columns=["Failure", "Pattern"],
)
alignment_failures

# 26. Computational Cost

Recurrent attention scores every source position for every target step. Alignment cost grows approximately with:

$$target\_length 	imes source\_length$$

In [ ]:
cost_factors = pd.DataFrame(
    [
        ("Source length", "more keys and values"),
        ("Target length", "more attention queries"),
        ("Attention dimension", "larger scoring network"),
        ("Beam width", "more active decoder hypotheses"),
        ("Batch size", "more simultaneous alignments"),
    ],
    columns=["Factor", "Effect"],
)
cost_factors

# 27. Attention Is Not Automatically an Explanation

Attention weights expose part of the model's computation, but they are not guaranteed to be faithful explanations because:

- alternative attention patterns may produce similar outputs;
- weights may vary across random seeds;
- later layers transform the context;
- high weight does not prove causal importance.

In [ ]:
interpretation_guidelines = pd.DataFrame(
    [
        ("Use", "inspect focus and alignment patterns"),
        ("Compare", "examples, models, and seeds"),
        ("Validate", "with perturbation or counterfactual tests"),
        ("Avoid", "claiming direct human-like reasoning"),
    ],
    columns=["Action", "Guideline"],
)
interpretation_guidelines

# 28. Arabic and Multilingual Considerations

Arabic attention patterns are affected by:

- attached proclitics and enclitics;
- rich inflection;
- optional tashkeel;
- orthographic variants;
- word-order differences;
- agreement;
- MSA, dialects, and code-switching.

In [ ]:
arabic_examples = pd.DataFrame(
    [
        ("وَسَيَكْتُبُونَهَا", "وَ + سَ + يَكْتُبُونَ + هَا"),
        ("بِالْمَدْرَسَةِ", "بِ + الْمَدْرَسَةِ"),
        ("كِتَابُهُمَا", "كِتَابُ + هُمَا"),
    ],
    columns=["Fully vocalized form", "Illustrative segmentation"],
)
arabic_examples

A single Arabic word may align with several target words. Word, subword, morphological, and character tokenization produce different alignment granularities.

In [ ]:
granularity_effects = pd.DataFrame(
    [
        ("Word", "coarse alignment", "large vocabulary"),
        ("Morphological segment", "clitic-aware alignment", "analyzer required"),
        ("Subword", "shared fragments", "fragmented interpretation"),
        ("Character", "fine alignment", "long sequences"),
    ],
    columns=["Unit", "Alignment effect", "Trade-off"],
)
granularity_effects

For fully vocalized Arabic tasks, tashkeel should be preserved when it is part of the task definition. Removing it changes token identities and possible alignments.

# 29. Reproducibility and Reporting

Report the corpus, split, tokenization, vocabularies, encoder directionality, model dimensions, score type, teacher-forcing schedule, optimizer, clipping, early stopping, decoding method, maximum length, metrics, seeds, and hardware.

In [ ]:
import platform

experiment_metadata = pd.Series(
    {
        "sentence_pairs": len(dataset),
        "training_pairs": len(train_frame),
        "validation_pairs": len(validation_frame),
        "test_pairs": len(test_frame),
        "source_vocabulary": len(source_vocabulary),
        "target_vocabulary": len(target_vocabulary),
        "encoder": "bidirectional GRU",
        "attention": "Bahdanau additive",
        "decoder": "GRUCell",
        "encoder_hidden_per_direction": ENCODER_HIDDEN_DIM,
        "decoder_hidden": DECODER_HIDDEN_DIM,
        "device": str(DEVICE),
        "random_seed": 42,
        "python_version": platform.python_version(),
        "numpy_version": np.__version__,
        "torch_version": torch.__version__,
    },
    name="Attention experiment",
)
experiment_metadata

# 30. Knowledge Check

1. What problem does attention address?  
2. What are queries, keys, and values?  
3. Why is masked softmax required?  
4. How is a context vector calculated?  
5. How do dot-product and additive attention differ?  
6. Why is attention called soft alignment?  
7. What does a bidirectional encoder produce?  
8. How does the decoder use the context?  
9. What shape does an alignment matrix have?  
10. What does attention entropy measure?  
11. Why may a concentrated alignment still be wrong?  
12. Why is attention not automatically an explanation?  
13. How does Arabic segmentation affect alignment granularity?

# 31. Exercises

## Exercise 1 — Masked Softmax
Implement masked softmax in PyTorch and test padded positions.

## Exercise 2 — Score Functions
Compare dot-product, general, and additive attention.

## Exercise 3 — Attention Dimensions
Train models with several attention dimensions.

## Exercise 4 — Encoder Directionality
Compare unidirectional and bidirectional encoders.

## Exercise 5 — Alignment Visualization
Visualize correct and incorrect translations.

## Exercise 6 — Entropy
Compare attention entropy for correct and incorrect outputs.

## Exercise 7 — Beam Search
Implement beam search with length normalization.

## Exercise 8 — No-Attention Baseline
Compare against a fixed-vector encoder–decoder.

## Exercise 9 — Arabic Translation
Build a fully vocalized Arabic sequence task and inspect alignments.

## Exercise 10 — Robustness
Add noise, unseen words, and longer sequences.

## Challenge Exercises

1. Add a coverage penalty.  
2. Implement monotonic attention.  
3. Compare attention with gradient-based attribution.  
4. Add established BLEU, chrF, and COMET implementations.  
5. Refactor the model into reusable training modules.

# 32. Summary and Next Lesson

In this lesson:

- attention was introduced as dynamic source access;
- queries, keys, values, scores, weights, and contexts were defined;
- padding-aware softmax was implemented;
- dot-product, general, and additive attention were distinguished;
- a bidirectional GRU encoder produced contextual source states;
- Bahdanau attention scored source positions;
- an attentive recurrent decoder generated target tokens;
- a CPU-only translation model was trained with PyTorch;
- greedy decoding returned translations and alignment matrices;
- attention heatmaps and entropy were analyzed;
- exact match, token accuracy, and BLEU-like scores were calculated;
- alignment failure modes and interpretation limits were examined;
- Arabic morphology, segmentation, and tashkeel were connected to attention granularity.

## Next Lesson

**Lesson 31: Self-Attention and Transformer Architecture Foundations** introduces scaled dot-product self-attention, multi-head attention, positional information, residual connections, layer normalization, and Transformer encoder blocks.

# References

- Bahdanau, D., Cho, K., & Bengio, Y. Neural machine translation with alignment.  
- Luong, M.-T., Pham, H., & Manning, C. D. Effective attention approaches.  
- Sutskever, I., Vinyals, O., & Le, Q. V. Sequence-to-sequence learning.  
- Jain, S., & Wallace, B. C. Attention and interpretability literature.  
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.